# Train a lesion classifier

Run the cells in order to train a model from raw MILK10k data and classify a curated photo.
Choose **padded crop** for melanoma vs. nevus or **direct 11-class** for all MILK10k diagnoses.

[Setup and checkpoints](../classifiers/README.md) · [Experiment findings](../docs/classifier-experiments.md)

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
import torch

cwd = Path.cwd().resolve()
repo_root = next(path for path in (cwd, *cwd.parents) if (path / "classifiers/train.py").is_file())
sys.path.insert(0, str(repo_root))

from classifiers.model import DermFMClassifier, Task
from classifiers.train import TrainingConfig, default_device, evaluate_model, extract_features, fit_model, prepare_data, save_model
from demo.samples import BUNDLED_SAMPLE_DIR

torch.set_num_threads(min(4, torch.get_num_threads()))

## 1. Configure

Set the task and paths. Padded crop needs the U-Net checkpoint. The defaults below match the selected training recipes.

In [ ]:
DATA_DIR = Path(os.environ.get("MILK10K_DATA_DIR", repo_root / "data")).expanduser()
TASK: Task = "padded_crop"  # "padded_crop" or "direct_11"
SEGMENTATION_MODEL = repo_root / "models/unet_segmentation_full.pth"
OUTPUT_ROOT = DATA_DIR / "models"
SAMPLE_PHOTO = BUNDLED_SAMPLE_DIR / "blue-nevus.jpg"

config = TrainingConfig(
    data_dir=DATA_DIR,
    task=TASK,
    device=default_device(),
    catalog=None,  # Or Path("final_data.csv") to use an existing catalog
    segmentation_model=SEGMENTATION_MODEL if TASK == "padded_crop" else None,
    batch_size=12,
    calibration_fraction=0.20,
    calibration_seed=42,
    sensitivity_target=0.95,
    regularization_c=1.0,
)

## 2. Split the data

Create or verify the frozen split. Both views of a lesion and exact duplicates stay together.
The calibration groups come from the development folds.

In [ ]:
data = prepare_data(config)
partition = data.rows["split"].replace({"train": "fit", "test": "holdout"}).mask(data.calibration, "calibration")
pd.crosstab(data.rows["diagnosis_code"], partition, margins=True)

## 3. Prepare photos and extract features

The direct model uses the original clinical photo. The padded model crops around the lesion with surrounding skin,
then uses the original frame if the mask is empty. Both use the same frozen DermFM encoder.

In [ ]:
features, provenance = extract_features(config, data)
print(f"{features.shape[0]:,} photos × {features.shape[1]:,} features on {config.device}")
print(f"Reused cached features: {provenance['feature_cache_hit']}")

## 4. Fit and calibrate

Each of the three heads trains on two development folds. The ensemble sets its melanoma cutoff using the reserved calibration groups.

In [ ]:
model = fit_model(config, data, features)
print(f"Melanoma cutoff: {model.cutoff:.6g}")

## 5. Evaluate the holdout

Use the locked cutoff to see how many melanomas the model flagged and how many other lesions it cleared.
The table shows per-diagnosis results and the sample counts behind them.

In [ ]:
reports, predictions = evaluate_model(config, data, features, model)
holdout = reports["holdout"]

for metric in ("melanoma_sensitivity", "melanoma_specificity"):
    result = holdout[metric]
    print(f"{metric.replace('_', ' ').capitalize()} {result['count']}/{result['total']} ({result['rate']:.1%})")

pd.DataFrame(holdout["classification_report"]).T.drop(index="accuracy", errors="ignore").round(3)

## 6. Save the model

Save the fitted heads, cutoff, and evaluation in a new folder. A padded-crop folder can also be passed to the demo with `--model`.

In [ ]:
model_dir = save_model(config, data, model, provenance, reports, predictions, OUTPUT_ROOT)
print(model_dir)

## 7. Reload and classify a photo

Set `SAMPLE_PHOTO` to a curated JPEG or PNG, or install the sample gallery from the setup instructions.
Set `model_dir` to an existing model folder to reuse an earlier run.

In [ ]:
if SAMPLE_PHOTO.is_file():
    classifier = DermFMClassifier(
        model_dir, config.data_dir, config.device, expected_task=config.task, segmentation_model=config.segmentation_model
    )
    print(classifier.predict_file(SAMPLE_PHOTO)["research_prediction"])
else:
    print(f"Set SAMPLE_PHOTO to an existing curated JPEG or PNG. Current path: {SAMPLE_PHOTO}")